# Práctica — Clase 10: la CNN 3D

Completa los dos ejercicios. Cada uno tiene celdas con `# TODO:` y una solución colapsada debajo.

In [ ]:
import numpy as np
import tensorflow as tf

def generar_volumen_sintetico(num_cortes=40, alto=128, ancho=128, con_nodulo=True, semilla=None):
    rng = np.random.default_rng(semilla)
    volumen = rng.normal(loc=0.2, scale=0.05, size=(num_cortes, alto, ancho)).astype("float32")
    if con_nodulo:
        centro = (num_cortes // 2, alto // 2, ancho // 2)
        radio = 8
        z, yy, xx = np.ogrid[:num_cortes, :alto, :ancho]
        mascara = (z - centro[0]) ** 2 + (yy - centro[1]) ** 2 + (xx - centro[2]) ** 2 <= radio ** 2
        volumen[mascara] += 0.6
    return np.clip(volumen, 0, 1)

def redimensionar_volumen(volumen, tamano_destino=(64, 64, 32)):
    volumen = tf.expand_dims(volumen, axis=-1)
    volumen = tf.image.resize(volumen, tamano_destino[:2])
    volumen = tf.transpose(volumen, perm=[2, 0, 1, 3])
    volumen = tf.image.resize(volumen, [tamano_destino[2], tamano_destino[0]])
    volumen = tf.transpose(volumen, perm=[1, 2, 0, 3])
    return volumen

volumenes, etiquetas = [], []
for i in range(40):
    con_nodulo = i % 2 == 0
    v = redimensionar_volumen(generar_volumen_sintetico(con_nodulo=con_nodulo, semilla=i))
    volumenes.append(v.numpy())
    etiquetas.append(1 if con_nodulo else 0)
X, y = np.array(volumenes), np.array(etiquetas)

## Ejercicio 1 — Completa la CNN 3D

**Enunciado**: define un modelo `Sequential` con una capa `Input` de forma `(64, 64, 32, 1)`, una capa `Conv3D` de 8 filtros (`kernel_size=3`, `relu`), una capa `MaxPooling3D`, `GlobalAveragePooling3D`, y la capa de salida para clasificación binaria.

**Pista**: `GlobalAveragePooling3D` reduce todo el volumen a un solo vector, listo para la capa densa final.

In [ ]:
modelo = tf.keras.Sequential([
    # TODO: añade una capa Input con shape=(64, 64, 32, 1)
    # TODO: añade una capa Conv3D de 8 filtros, kernel_size=3, activation='relu'
    # TODO: añade una capa MaxPooling3D
    # TODO: añade una capa GlobalAveragePooling3D
    # TODO: añade la capa Dense de salida para clasificación binaria
])

modelo.compile(optimizer="adam",
               loss="binary_crossentropy",
               metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

modelo.fit(X, y, epochs=5, batch_size=4)

<details>
<summary>💡 Ver solución</summary>

```python
modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(64, 64, 32, 1)),
    tf.keras.layers.Conv3D(8, kernel_size=3, activation="relu"),
    tf.keras.layers.MaxPooling3D(),
    tf.keras.layers.GlobalAveragePooling3D(),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
```
</details>

## Ejercicio 2 — Añade la métrica AUC

**Enunciado**: completa `modelo.compile` para que, además de `accuracy`, incluya la métrica `tf.keras.metrics.AUC(name="auc")`, tal como hace la evaluación real de la competición de Kaggle.

**Pista**: el parámetro `metrics` acepta una lista de varias métricas a la vez.

In [ ]:
modelo_v2 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(64, 64, 32, 1)),
    tf.keras.layers.Conv3D(8, kernel_size=3, activation="relu"),
    tf.keras.layers.MaxPooling3D(),
    tf.keras.layers.GlobalAveragePooling3D(),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

# TODO: compila modelo_v2 con optimizer='adam', loss='binary_crossentropy',
# y metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]

modelo_v2.fit(X, y, epochs=5, batch_size=4)

<details>
<summary>💡 Ver solución</summary>

```python
modelo_v2.compile(optimizer="adam",
                  loss="binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
```
</details>